# 🤖 Discord AI Agent on Google Colab (Groq LPU)
An asynchronous, permission-aware Discord AI Agent powered by **Groq LPU inference** (`openai/gpt-oss-20b` or `qwen/qwen3.8-27b`) and **discord.py**.

---

### 🔑 Credentials Setup:
You can configure credentials in two ways:
1. **Option A (Recommended):** Use **Google Colab Secrets** (click the 🔑 key icon on the left sidebar):
   - `DISCORD_BOT_TOKEN` — Bot token from Discord Developer Portal -> Bot -> Reset Token
   - `GROQ_API_KEY` — API Key from https://console.groq.com/keys
   - `DISCORD_GUILD_ID` — Discord Server ID (Developer Mode -> Right-click server -> Copy ID)
   *(Make sure **Notebook access** toggle is enabled for all 3 secrets)*
2. **Option B:** Directly paste them into **Cell 2** below.

## 📦 Part 1: Install Core Dependencies
Installs `discord.py` (Discord WebSocket API), `groq` (LPU inference), and `nest_asyncio` (enables nested event loops in Colab).

In [ ]:
# Cell 1: Install dependencies
!pip install -q -U discord.py groq nest_asyncio
print("✅ Core libraries installed successfully!")

## 🔑 Part 2: Credentials & Configuration
Reads credentials from Colab Secrets or uses fallback direct variables.

In [ ]:
# Cell 2: Configuration & Credentials
import os

# 1. Attempt reading from Colab Secrets (Left sidebar 🔑)
try:
    from google.colab import userdata
    DISCORD_BOT_TOKEN = userdata.get("DISCORD_BOT_TOKEN")
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    DISCORD_GUILD_ID = int(userdata.get("DISCORD_GUILD_ID"))
except Exception:
    # 2. Or paste directly below:
    DISCORD_BOT_TOKEN = "your_discord_bot_token_here"
    GROQ_API_KEY = "your_groq_api_key_here"
    DISCORD_GUILD_ID = 123456789012345678

# Selected high-speed model on Groq
MODEL = "openai/gpt-oss-20b"  # Alternative: "qwen/qwen3.8-27b"

print("✅ Configuration and credentials loaded!")

## 🔍 Part 3: Automated Pre-Flight Health Check
Runs a 3-second diagnostic test to verify Groq API inference, Discord Gateway connection, and server access before starting the bot.

In [ ]:
# Cell 3: Pre-Flight Verification
import asyncio
import contextlib
import discord
from groq import AsyncGroq
import nest_asyncio

nest_asyncio.apply()

async def run_diagnostics():
    print("=" * 60)
    print("🔍 RUNNING PRE-FLIGHT DIAGNOSTICS...")
    print("=" * 60)
    all_ok = True

    # Check 1: Groq API Inference
    print("\n[1/3] Testing GroqCloud API connection...")
    try:
        groq_client = AsyncGroq(api_key=GROQ_API_KEY)
        resp = await groq_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Respond with: OK"}],
            max_completion_tokens=10,
        )
        print(f"  ✅ Groq API PASSED: Model '{MODEL}' is active and ready.")
    except Exception as e:
        all_ok = False
        print(f"  ❌ Groq API FAILED: {type(e).__name__} - {e}")

    # Check 2: Discord Gateway & Token
    print("\n[2/3] Testing Discord Token & Gateway connection...")
    intents = discord.Intents.default()
    intents.message_content = True
    client = discord.Client(intents=intents)
    ready = asyncio.Event()

    @client.event
    async def on_ready():
        ready.set()

    login_task = asyncio.create_task(client.start(DISCORD_BOT_TOKEN))
    discord_ok = False

    try:
        await asyncio.wait_for(ready.wait(), timeout=10.0)
        discord_ok = True
        print(f"  ✅ Discord Token PASSED: Logged in as: '{client.user}'")
    except asyncio.TimeoutError:
        all_ok = False
        print("  ❌ Discord FAILED: Connection timed out. Check your DISCORD_BOT_TOKEN.")
    except discord.errors.PrivilegedIntentsRequired:
        all_ok = False
        print("  ❌ Discord FAILED: Turn ON 'Message Content Intent' in Developer Portal!")
    except Exception as e:
        all_ok = False
        print(f"  ❌ Discord FAILED: {type(e).__name__} - {e}")

    # Check 3: Guild Membership
    if discord_ok and client.is_ready():
        print("\n[3/3] Checking Discord Server Membership...")
        guild = client.get_guild(int(DISCORD_GUILD_ID))
        if guild:
            print(f"  ✅ Server PASSED: Found '{guild.name}' ({len(guild.text_channels)} text channels)")
        else:
            all_ok = False
            print(f"  ❌ Server FAILED: Bot is not invited to Server ID {DISCORD_GUILD_ID}.")

    # Cleanup socket
    if not client.is_closed():
        await client.close()
    login_task.cancel()
    with contextlib.suppress(Exception):
        await login_task

    print("\n" + "=" * 60)
    if all_ok:
        print("🎉 ALL CHECKS PASSED! Ready to start the live bot below.")
    else:
        print("⚠️ SOME CHECKS FAILED. Please review the errors above.")
    print("=" * 60)

await run_diagnostics()

## 🛡️ Part 4: Re-Run Guard & Bot Client Setup
Gracefully terminates any previous background bot sessions in Colab to prevent duplicate replies, and enables instant guild slash command syncing.

In [ ]:
# Cell 4: Re-Run Guard & Client Initialization
import asyncio
import io
import discord
from discord import app_commands
from discord.ext import commands
from groq import AsyncGroq
import nest_asyncio

nest_asyncio.apply()

# 1. Cleanly close previous bot session
for bot_key in ["_bot", "_skillup_bot", "_live_bot"]:
    b = globals().get(bot_key)
    if b and not b.is_closed():
        print(f"⏳ Closing lingering bot connection ({bot_key})...")
        await b.close()

for task_key in ["_task", "_skillup_task", "_live_task"]:
    t = globals().get(task_key)
    if t and not t.done():
        print(f"⏳ Cancelling background task ({task_key})...")
        t.cancel()
        try:
            await t
        except (asyncio.CancelledError, Exception):
            pass

# 2. Configure Intents and Target Guild
intents = discord.Intents.default()
intents.message_content = True
GUILD = discord.Object(id=DISCORD_GUILD_ID)
llm = AsyncGroq(api_key=GROQ_API_KEY)

# 3. Custom Bot Class with Instant Guild Syncing
class SkillUpBot(commands.Bot):
    async def setup_hook(self):
        self.tree.copy_global_to(guild=GUILD)
        synced = await self.tree.sync(guild=GUILD)
        print("⚡ Slash commands registered to server:", [cmd.name for cmd in synced])

bot = SkillUpBot(command_prefix="!", intents=intents)
globals()["_bot"] = bot
print("✅ Bot client initialized and ready!")

## 🧠 Part 5: Groq Inference Function & Command Handlers
Implements:
- `ask_groq`: Asynchronous non-blocking LPU query.
- `on_ready`: Status confirmation.
- `on_message`: Direct channel `@mention` support with live typing indicators.
- `/ping`: Instant WebSocket latency check.
- `/agent`: AI Q&A with Discord 3-second timeout protection (`defer`) and 2,000-character payload handling.

In [ ]:
# Cell 5: Groq Query Helper and Commands

# --- A. Groq Inference Function ---
async def ask_groq(system_message: str, user_message: str, tokens: int = 700) -> str:
    kwargs = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        "temperature": 0.6,
        "max_completion_tokens": tokens,
    }
    response = await llm.chat.completions.create(**kwargs)
    return (response.choices[0].message.content or "No response generated.").strip()

# --- B. Connection Event ---
@bot.event
async def on_ready():
    print(f"\n🚀 ONLINE: {bot.user} | Connected to Server ID: {DISCORD_GUILD_ID}")
    print("👉 Go to your Discord server and test /ping or @mention!")

# --- C. Direct @Mention Listener ---
@bot.event
async def on_message(message: discord.Message):
    if message.author == bot.user or message.author.bot:
        return

    if bot.user in message.mentions:
        clean_text = message.clean_content.replace(f"@{bot.user.display_name}", "").strip()
        if not clean_text:
            await message.reply("Hello! How can I assist you? Ask me a question or try `/agent <prompt>`.")
            return

        async with message.channel.typing():
            try:
                answer = await ask_groq(
                    "You are a helpful workshop AI assistant. Answer clearly and concisely.",
                    clean_text[:3500],
                    tokens=850,
                )
                if len(answer) <= 1850:
                    await message.reply(answer, allowed_mentions=discord.AllowedMentions.none())
                else:
                    report = discord.File(io.BytesIO(answer.encode("utf-8")), "answer.txt")
                    await message.reply("Response was long, attached as file:", file=report)
            except Exception as err:
                print(f"Error in on_message: {type(err).__name__} - {err}")
                await message.reply(f"❌ Error: {type(err).__name__}: {err}")

    await bot.process_commands(message)

# --- D. Slash Command: /ping ---
@bot.tree.command(name="ping", description="Check if the bot is online")
async def ping(interaction: discord.Interaction):
    await interaction.response.send_message("🏓 Pong! Your SkillUp bot is online and working!")

# --- E. Slash Command: /agent ---
@bot.tree.command(name="agent", description="Ask the Groq-powered AI assistant a question")
@app_commands.describe(prompt="Your question or instruction for the AI")
async def agent(interaction: discord.Interaction, prompt: str):
    # 1. Defeat Discord's 3-second timeout rule
    await interaction.response.defer(thinking=True)
    try:
        # 2. Query Groq
        answer = await ask_groq(
            "You are a friendly, concise workshop AI assistant. Explain technical concepts clearly.",
            prompt[:3500],
            tokens=850,
        )
        # 3. Guard against Discord 2,000-character limit
        if len(answer) <= 1850:
            await interaction.followup.send(answer, allowed_mentions=discord.AllowedMentions.none())
        else:
            report = discord.File(io.BytesIO(answer.encode("utf-8")), "agent_answer.txt")
            await interaction.followup.send("Response was over 1850 chars; attached below:", file=report)
    except Exception as err:
        print(f"Error in /agent: {type(err).__name__} - {err}")
        await interaction.followup.send(f"❌ AI Error: {type(err).__name__}: {err}", ephemeral=True)

print("✅ Event handlers and slash commands registered!")

## 🚀 Part 6: Start the Bot in Background
Spawns `bot.start()` as an asynchronous background task so the Colab notebook stays responsive.

In [ ]:
# Cell 6: Start Bot Background Task
async def start_discord_bot():
    try:
        await bot.start(DISCORD_BOT_TOKEN)
    except discord.errors.LoginFailure:
        print("❌ Login Failed: DISCORD_BOT_TOKEN is invalid. Reset token in Developer Portal.")
    except Exception as e:
        print(f"❌ Bot encountered error: {type(e).__name__} - {e}")

task = asyncio.create_task(start_discord_bot())
globals()["_task"] = task
print("Connecting to Discord Gateway... check output logs below:")